####  Bronze vs Silver Testing – vouchers Table

This document describes the data quality, integrity, and reconciliation tests performed for the `vouchers` table during ingestion from Bronze to Silver.





####  Tables Under Test

- Bronze: `coffee.bronze.vouchers`
- Silver: `coffee.silver.vouchers`


In [0]:
-- Test 1: Compare row counts between Bronze and Silver
-- Silver count should be <= Bronze count due to filtering and deduplication

SELECT 'bronze' AS layer, COUNT(*) AS record_count
FROM coffee.bronze.vouchers

UNION ALL

SELECT 'silver' AS layer, COUNT(*) AS record_count
FROM coffee.silver.vouchers;

In [0]:
-- Test 2: Ensure mandatory columns are NOT NULL in Silver
-- voucher_id, discount_type, discount_value, valid_from, valid_to must be present

SELECT COUNT(*) AS invalid_silver_records
FROM coffee.silver.vouchers
WHERE
  voucher_id IS NULL
  OR discount_type IS NULL
  OR discount_value IS NULL
  OR valid_from IS NULL
  OR valid_to IS NULL;


In [0]:
-- Test 3: Ensure voucher validity periods are logically correct
-- valid_to should always be greater than or equal to valid_from

SELECT COUNT(*) AS invalid_validity_periods
FROM coffee.silver.vouchers
WHERE valid_to < valid_from;


In [0]:
-- Test 4: Ensure composite business key uniqueness
-- (voucher_id, valid_from, valid_to) defines a unique record

SELECT voucher_id, valid_from, valid_to, COUNT(*) AS cnt
FROM coffee.silver.vouchers
GROUP BY voucher_id, valid_from, valid_to
HAVING COUNT(*) > 1;


In [0]:
-- Test 6: All valid Bronze voucher records should be present in Silver
-- Identifies valid records accidentally dropped

SELECT voucher_id, valid_from, valid_to
FROM coffee.bronze.vouchers
WHERE
  voucher_id IS NOT NULL
  AND discount_type IS NOT NULL
  AND discount_value IS NOT NULL
  AND valid_from IS NOT NULL
  AND valid_to IS NOT NULL

EXCEPT

SELECT voucher_id, valid_from, valid_to
FROM coffee.silver.vouchers;
